# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**READING THE DATASET**

In [1]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN '')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [2]:
table = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Lane: Refresh / Content Opportunity Scoring → a ranking problem.** Per the
`training-honest-models` skill's method table, "which first?" ranking calls for a classifier's
probability output evaluated at precision@K, not a hard classification metric — matches every
prior notebook's reasoning (reviewer capacity is the constraint, not overall accuracy).

**Methods, in order of complexity:** Logistic Regression first (readable coefficients, a
natural step up from the depth-2/3 decision trees already built), then Random Forest (handles
the feature interactions ML-06 already showed exist — no single signal cleanly separated
declining pages). Gradient Boosting is left out this round: ML-06 also surfaced a real flaw in
the label definition (zero-click Feb pages can't mechanically register as "declining"), and per
the skill's own standard — read the errors before believing the score — that gets fixed first
rather than layering a stronger model on top of a known-bad label.

**Months used: Feb 2026 (features) → March 2026 (label).** Confirmed mid-panel, not the sealed
test month — per `flyrank-data/SKILL.md`, `fact_content_daily_performance_sample` is
specifically June 2026 and is off-limits for label development; Feb/March are two ordinary
partitions of the full 79M-row table, not that table.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, numpy as np, pandas as pd

dim_clients_table = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"
dim_content_table  = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"  # for later, not used yet

# Re-run the grain probe with the corrected path
print("dim_clients grain check (should be empty):")
print(con.sql(f"""
    SELECT client_hash_id, COUNT(*) AS n
    FROM read_parquet('{dim_clients_table}')
    GROUP BY client_hash_id HAVING COUNT(*) > 1 LIMIT 5
"""))

print("\ndim_clients history-start distribution:")
print(con.sql(f"""
    SELECT MIN(gsc_data_start) AS earliest_gsc, MAX(gsc_data_start) AS latest_gsc,
           MIN(ga4_data_start) AS earliest_ga4, MAX(ga4_data_start) AS latest_ga4,
           COUNT(*) AS n_clients
    FROM read_parquet('{dim_clients_table}')
"""))


dim_clients grain check (should be empty):
┌────────────────┬───────┐
│ client_hash_id │   n   │
│    varchar     │ int64 │
├────────────────┴───────┤
│         0 rows         │
└────────────────────────┘


dim_clients history-start distribution:
┌──────────────┬────────────┬──────────────┬────────────┬───────────┐
│ earliest_gsc │ latest_gsc │ earliest_ga4 │ latest_ga4 │ n_clients │
│     date     │    date    │     date     │    date    │   int64   │
├──────────────┼────────────┼──────────────┼────────────┼───────────┤
│ 2025-01-27   │ 2026-06-02 │ 2025-10-29   │ 2026-06-01 │       104 │
└──────────────┴────────────┴──────────────┴────────────┴───────────┘



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Client-grouped train/test split** — same reasoning as the starter-CSV work (a client's pages
never appear in both train and test), now applied to the real warehouse. Also filtering to
clients whose `gsc_data_start` is on or before Feb 1, 2026 — per the panel warning, history
depth differs wildly per client, and a client that only started mid-February would have a
partial, misleading Feb feature window rather than a genuinely missing one.

This is grouped, not time-aware, in the strict sense used for time-series backtests — the
feature/label windows (Feb → March) are already time-ordered and non-overlapping, so the
temporal honesty is handled by the label design itself; the split's job is purely to stop a
client's own pages from leaking between train and test.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
RANDOM_STATE = 42
FEATURE_MONTH, LABEL_MONTH = "2026-02", "2026-03"

eligible_clients = con.sql(f"""
    SELECT client_hash_id
    FROM read_parquet('{dim_clients_table}')
    WHERE gsc_data_start <= DATE '2026-02-01'
""").df()["client_hash_id"].tolist()

print(f"Clients with full Feb history: {len(eligible_clients)}")

rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(eligible_clients)
n_test = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test])
train_clients = set(shuffled[n_test:])
print(f"Train clients: {len(train_clients)}  |  Test clients: {len(test_clients)}")

Clients with full Feb history: 41
Train clients: 33  |  Test clients: 8


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Fixing the label before trusting any score.** ML-06 found that `clicks_mar < clicks_feb`
mechanically floors decline rate at 0 for any zero-click-Feb page (52% of the panel) — not a
real signal, a label artifact. Corrected here two ways: restrict to pages with real Feb clicks
(`clicks_feb > 0`, so decline is actually possible), and require a meaningful drop
(`clicks_mar < 0.8 * clicks_feb`), not any decrease, so single-click noise doesn't count.

**Same data, same metric, same split for all three rows below:** the Week-4 rule baseline
(`visible_but_low_rank`, recomputed here on the corrected label so the comparison is apples to
apples) is scored on the exact same held-out test-client rows as Logistic Regression and Random
Forest — none of the three sees any advantage from a different eligible set.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

panel = con.sql(f"""
    WITH feb AS (
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_avg_position)      AS avg_position_feb,
               SUM(gsc_impressions)       AS impressions_feb,
               SUM(gsc_clicks)            AS clicks_feb,
               SUM(ga4_sessions)          AS ga4_sessions_feb,
               SUM(ga4_engaged_sessions)  AS ga4_engaged_sessions_feb,
               SUM(ga4_pageviews)         AS ga4_pageviews_feb,
               SUM(scroll_events)         AS scroll_events_feb,
               SUM(sessions_ai)           AS sessions_ai_feb,
               bool_and(ga4_data_available) AS ga4_available_all_feb
        FROM read_parquet('{fact_table}') WHERE month = '{FEATURE_MONTH}'
        GROUP BY client_hash_id, content_hash_id
    ), mar AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_mar
        FROM read_parquet('{fact_table}') WHERE month = '{LABEL_MONTH}'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT feb.*, mar.clicks_mar
    FROM feb JOIN mar USING (client_hash_id, content_hash_id)
    WHERE feb.clicks_feb > 0
""").df()

print(f"Eligible panel (clicks_feb > 0): {len(panel)} rows")

panel["is_declining"] = (panel["clicks_mar"] < 0.8 * panel["clicks_feb"]).astype(int)
panel["ctr_feb"] = panel["clicks_feb"] / panel["impressions_feb"].replace(0, np.nan)
panel["had_ga4_feb"] = panel["ga4_sessions_feb"] > 0
panel["engagement_rate_feb"] = np.where(panel["had_ga4_feb"], panel["ga4_engaged_sessions_feb"] / panel["ga4_sessions_feb"], 0.0)
panel["scroll_rate_feb"] = np.where(panel["ga4_pageviews_feb"] > 0, panel["scroll_events_feb"] / panel["ga4_pageviews_feb"], 0.0)
panel["ai_referral_share_feb"] = np.where(panel["had_ga4_feb"], panel["sessions_ai_feb"] / panel["ga4_sessions_feb"], 0.0)

# Fill the boolean column with False (its own dtype's valid fill), everything else with 0
panel["ga4_available_all_feb"] = panel["ga4_available_all_feb"].fillna(False)
numeric_cols = panel.select_dtypes(include=[np.number]).columns
panel[numeric_cols] = panel[numeric_cols].fillna(0)

# Baseline rule, recomputed on this corrected panel (median split within eligible panel)
median_impr = panel["impressions_feb"].median()
panel["baseline_flag"] = ((panel["impressions_feb"] >= median_impr) & (panel["avg_position_feb"] > 10)).astype(int)
panel["baseline_score"] = panel["baseline_flag"] * panel["impressions_feb"]

train_mask = panel["client_hash_id"].isin(train_clients)
test_mask = panel["client_hash_id"].isin(test_clients)

feature_cols = ["avg_position_feb", "impressions_feb", "ctr_feb", "engagement_rate_feb", "scroll_rate_feb", "ai_referral_share_feb"]
X_train, X_test = panel.loc[train_mask, feature_cols], panel.loc[test_mask, feature_cols]
y_train, y_test = panel.loc[train_mask, "is_declining"].values, panel.loc[test_mask, "is_declining"].values
baseline_test_scores = panel.loc[test_mask, "baseline_score"].values

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=RANDOM_STATE).fit(X_train, y_train)

lr_scores = lr.predict_proba(X_test)[:, 1]
rf_scores = rf.predict_proba(X_test)[:, 1]

print(f"\nTest-set base rate (declining): {y_test.mean():.1%}  |  n_test = {len(y_test)}\n")
print(f"{'Model':<20}{'Precision@20':>15}{'Precision@50':>15}")
for name, scores in [("Baseline rule", baseline_test_scores), ("Logistic Regression", lr_scores), ("Random Forest", rf_scores)]:
    print(f"{name:<20}{precision_at_k(scores, y_test, 20):>15.3f}{precision_at_k(scores, y_test, 50):>15.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible panel (clicks_feb > 0): 53738 rows

Test-set base rate (declining): 50.6%  |  n_test = 1784

Model                  Precision@20   Precision@50
Baseline rule                 0.250          0.340
Logistic Regression           0.950          0.880
Random Forest                 0.900          0.900


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Reading the errors, not just the table above: which features Random Forest actually leans on
(sanity-checked — a suspiciously dominant top feature would mean a leak slipped through despite
the Feb-only construction), where the model is most wrong by position tier and volume, and three
concrete misses read by hand rather than just counted.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Random Forest feature importances:")
print(importances.round(3))
print("\nSanity check: no single feature should dominate near-total importance --",
      "that would suggest a leak even though every feature was built from Feb only.")

test_df = panel.loc[test_mask].copy()
test_df["rf_score"] = rf_scores
test_df["rf_pred"] = (rf_scores >= 0.5).astype(int)
test_df["correct"] = (test_df["rf_pred"] == test_df["is_declining"])

def tier(p):
    if p <= 3: return "top_3"
    if p <= 10: return "page_1"
    if p <= 20: return "page_2"
    return "deep"
test_df["position_tier"] = test_df["avg_position_feb"].apply(tier)

print("\nError rate by position tier:")
print(test_df.groupby("position_tier", observed=True)["correct"].agg(["size", "mean"]).rename(columns={"mean": "accuracy"}))

print("\n3 concrete wrong cases (false negatives -- model missed a real decline):")
misses = test_df[(test_df["is_declining"] == 1) & (test_df["rf_pred"] == 0)].sort_values("rf_score").head(3)
for _, row in misses.iterrows():
    print(f"  content={row['content_hash_id'][:12]}...  rf_score={row['rf_score']:.3f}  "
          f"avg_position={row['avg_position_feb']:.1f}  clicks_feb={row['clicks_feb']:.0f}  clicks_mar={row['clicks_mar']:.0f}")


Random Forest feature importances:
impressions_feb          0.547
ctr_feb                  0.277
avg_position_feb         0.106
scroll_rate_feb          0.039
engagement_rate_feb      0.025
ai_referral_share_feb    0.006
dtype: float64

Sanity check: no single feature should dominate near-total importance -- that would suggest a leak even though every feature was built from Feb only.

Error rate by position tier:
               size  accuracy
position_tier                
deep            196  0.658163
page_1         1224  0.592320
page_2          300  0.610000
top_3            64  0.609375

3 concrete wrong cases (false negatives -- model missed a real decline):
  content=content_8644...  rf_score=0.335  avg_position=6.0  clicks_feb=1  clicks_mar=0
  content=content_f1a1...  rf_score=0.348  avg_position=4.9  clicks_feb=7  clicks_mar=4
  content=content_9fb9...  rf_score=0.348  avg_position=13.7  clicks_feb=7  clicks_mar=3


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.